<a id="section1"></a>
## 1. Load Libraries and Models

In [ ]:
# Import Essential Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from pathlib import Path
import joblib
from datetime import datetime

# Machine Learning
from sklearn.preprocessing import RobustScaler, LabelEncoder
import xgboost as xgb

# Configuration
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("✓ All libraries imported successfully!")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"XGBoost version: {xgb.__version__}")

In [ ]:
# Load trained models
models_dir = Path('models')

if not models_dir.exists():
    print("⚠ ERROR: Models directory not found!")
    print("Please run Notebook 1 first to train and save the models.")
else:
    print("="*80)
    print("LOADING TRAINED MODELS")
    print("="*80)
    
    xgb_model = joblib.load(models_dir / 'xgboost_model.pkl')
    elastic_model = joblib.load(models_dir / 'elastic_net_model.pkl')
    scaler = joblib.load(models_dir / 'scaler.pkl')
    feature_info = joblib.load(models_dir / 'feature_info.pkl')
    model_params = joblib.load(models_dir / 'model_parameters.pkl')
    
    print("✓ XGBoost model loaded")
    print("✓ Elastic Net model loaded")
    print("✓ Scaler loaded")
    print("✓ Feature information loaded")
    
    print(f"\nModel Performance (from training):")
    print(f"  - XGBoost CV RMSE: {model_params['xgb_cv_rmse']}")
    print(f"  - Elastic Net CV RMSE: {model_params['elastic_cv_rmse']}")
    print(f"  - Expected features: {feature_info['n_features']}")

<a id="section2"></a>
## 2. Load and Preprocess Test Data

**Note:** The test data must go through the same preprocessing pipeline as the training data.

In [ ]:
# Load test data
data_path = Path('./dataset')
test_df = pd.read_csv(data_path / 'test.csv')

print("="*80)
print("TEST DATA LOADED")
print("="*80)
print(f"Test set shape: {test_df.shape}")
print(f"Number of houses to predict: {len(test_df)}")
print(f"\nFirst few rows:")
test_df.head()

In [ ]:
# Apply the same preprocessing as training data
# (This should match the preprocessing in Notebook 1)

print("="*80)
print("PREPROCESSING TEST DATA")
print("="*80)

test_processed = test_df.copy()

# 1. Missing value treatment
na_means_none = {
    'Alley': 'No_Alley',
    'BsmtQual': 'No_Basement',
    'BsmtCond': 'No_Basement',
    'BsmtExposure': 'No_Basement',
    'BsmtFinType1': 'No_Basement',
    'BsmtFinType2': 'No_Basement',
    'FireplaceQu': 'No_Fireplace',
    'GarageType': 'No_Garage',
    'GarageFinish': 'No_Garage',
    'GarageQual': 'No_Garage',
    'GarageCond': 'No_Garage',
    'PoolQC': 'No_Pool',
    'Fence': 'No_Fence',
    'MiscFeature': 'None',
    'GarageArea': 0,
    'GarageCars': 0,
    'BsmtFinSF1': 0,
    'BsmtFinSF2': 0,
    'BsmtUnfSF': 0,
    'TotalBsmtSF': 0,
    'BsmtFullBath': 0,
    'BsmtHalfBath': 0,
    'MasVnrArea': 0
}

for feature, fill_value in na_means_none.items():
    if feature in test_processed.columns:
        test_processed[feature].fillna(fill_value, inplace=True)

# Special handling for GarageYrBlt
if 'GarageYrBlt' in test_processed.columns and 'YearBuilt' in test_processed.columns:
    test_processed['GarageYrBlt'].fillna(test_processed['YearBuilt'], inplace=True)

# Fill remaining missing values
for col in test_processed.columns:
    if test_processed[col].isnull().sum() > 0:
        if test_processed[col].dtype == 'object':
            test_processed[col].fillna('Unknown', inplace=True)
        else:
            test_processed[col].fillna(test_processed[col].median(), inplace=True)

print(f"✓ Missing values handled: {test_processed.isnull().sum().sum()} remaining")

In [ ]:
# 2. Categorical encoding
ordinal_features = {
    'ExterQual': ['Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'ExterCond': ['Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'BsmtQual': ['No_Basement', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'BsmtCond': ['No_Basement', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'HeatingQC': ['Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'KitchenQual': ['Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'FireplaceQu': ['No_Fireplace', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'GarageQual': ['No_Garage', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'GarageCond': ['No_Garage', 'Po', 'Fa', 'TA', 'Gd', 'Ex']
}

for feature, categories in ordinal_features.items():
    if feature in test_processed.columns:
        mapping = {cat: idx for idx, cat in enumerate(categories)}
        test_processed[feature] = test_processed[feature].map(mapping)
        test_processed[feature].fillna(-1, inplace=True)

# One-hot encoding
categorical_features = test_processed.select_dtypes(include=['object']).columns.tolist()
if 'Id' in categorical_features:
    categorical_features.remove('Id')

low_cardinality = [col for col in categorical_features if test_processed[col].nunique() < 10]

if low_cardinality:
    test_encoded = pd.get_dummies(test_processed, columns=low_cardinality, drop_first=True)
    test_processed = test_encoded

# Label encoding for high cardinality
high_cardinality = [col for col in test_processed.select_dtypes(include=['object']).columns 
                   if col not in ['Id']]

for col in high_cardinality:
    if col in test_processed.columns:
        le = LabelEncoder()
        test_processed[col] = le.fit_transform(test_processed[col].astype(str))

print(f"✓ Categorical encoding complete")

In [ ]:
# 3. Feature engineering
def engineer_features(df):
    df_eng = df.copy()
    
    if all(col in df_eng.columns for col in ['TotalBsmtSF', '1stFlrSF', '2ndFlrSF']):
        df_eng['TotalSF'] = df_eng['TotalBsmtSF'] + df_eng['1stFlrSF'] + df_eng['2ndFlrSF']
    
    if all(col in df_eng.columns for col in ['BsmtFullBath', 'BsmtHalfBath', 'FullBath', 'HalfBath']):
        df_eng['TotalBath'] = df_eng['BsmtFullBath'] + 0.5*df_eng['BsmtHalfBath'] + df_eng['FullBath'] + 0.5*df_eng['HalfBath']
    
    if all(col in df_eng.columns for col in ['YrSold', 'YearBuilt']):
        df_eng['HouseAge'] = df_eng['YrSold'] - df_eng['YearBuilt']
        df_eng['HouseAge'] = df_eng['HouseAge'].clip(lower=0)
    
    if all(col in df_eng.columns for col in ['YrSold', 'YearRemodAdd']):
        df_eng['YearsSinceRemodel'] = df_eng['YrSold'] - df_eng['YearRemodAdd']
        df_eng['YearsSinceRemodel'] = df_eng['YearsSinceRemodel'].clip(lower=0)
    
    if all(col in df_eng.columns for col in ['OverallQual', 'TotalSF']):
        df_eng['QualityTimesSize'] = df_eng['OverallQual'] * df_eng['TotalSF']
    
    if all(col in df_eng.columns for col in ['GrLivArea', 'TotRmsAbvGrd']):
        df_eng['AreaPerRoom'] = df_eng['GrLivArea'] / (df_eng['TotRmsAbvGrd'] + 1)
    
    return df_eng

test_engineered = engineer_features(test_processed)
print(f"✓ Feature engineering complete")
print(f"Test data shape: {test_engineered.shape}")

In [ ]:
# 4. Prepare final feature matrix
X_test_final = test_engineered.drop(['Id'], axis=1, errors='ignore')

# Ensure all columns are numeric
for col in X_test_final.select_dtypes(include=['object']).columns:
    X_test_final[col] = pd.to_numeric(X_test_final[col], errors='coerce').fillna(0)

# Align with training features
expected_features = feature_info['feature_names']

# Add missing features
for feat in expected_features:
    if feat not in X_test_final.columns:
        X_test_final[feat] = 0

# Remove extra features
X_test_final = X_test_final[expected_features]

print("="*80)
print("PREPROCESSING COMPLETE")
print("="*80)
print(f"Final test shape: {X_test_final.shape}")
print(f"Expected features: {len(expected_features)}")
print(f"Match: {'✓ Yes' if X_test_final.shape[1] == len(expected_features) else '✗ No'}")

<a id="section3"></a>
## 3. Generate Predictions

In [ ]:
# Generate predictions with all models
print("="*80)
print("GENERATING PREDICTIONS")
print("="*80)

# XGBoost predictions
print("\nPredicting with XGBoost...")
predictions_xgb_log = xgb_model.predict(X_test_final)
predictions_xgb = np.expm1(predictions_xgb_log)  # Convert from log scale
print(f"✓ XGBoost predictions complete")

# Elastic Net predictions (requires scaling)
print("\nPredicting with Elastic Net...")
X_test_scaled = scaler.transform(X_test_final)
predictions_elastic_log = elastic_model.predict(X_test_scaled)
predictions_elastic = np.expm1(predictions_elastic_log)
print(f"✓ Elastic Net predictions complete")

# Ensemble (average of both models)
print("\nCreating ensemble predictions...")
predictions_ensemble = (predictions_xgb + predictions_elastic) / 2
print(f"✓ Ensemble predictions complete")

print(f"\n{'='*80}")
print("PREDICTION SUMMARY")
print("="*80)
print(f"Total predictions: {len(predictions_xgb)}")
print(f"\nXGBoost:")
print(f"  Mean: ${predictions_xgb.mean():,.0f}")
print(f"  Min: ${predictions_xgb.min():,.0f}")
print(f"  Max: ${predictions_xgb.max():,.0f}")
print(f"\nElastic Net:")
print(f"  Mean: ${predictions_elastic.mean():,.0f}")
print(f"  Min: ${predictions_elastic.min():,.0f}")
print(f"  Max: ${predictions_elastic.max():,.0f}")
print(f"\nEnsemble:")
print(f"  Mean: ${predictions_ensemble.mean():,.0f}")
print(f"  Min: ${predictions_ensemble.min():,.0f}")
print(f"  Max: ${predictions_ensemble.max():,.0f}")

<a id="section4"></a>
## 4. Create Submission Files

In [ ]:
# Create submission DataFrames
print("="*80)
print("CREATING SUBMISSION FILES")
print("="*80)

submission_xgb = pd.DataFrame({
    'Id': test_df['Id'],
    'SalePrice': predictions_xgb
})

submission_elastic = pd.DataFrame({
    'Id': test_df['Id'],
    'SalePrice': predictions_elastic
})

submission_ensemble = pd.DataFrame({
    'Id': test_df['Id'],
    'SalePrice': predictions_ensemble
})

# Save to CSV
submission_xgb.to_csv('./dataset/submission_xgboost.csv', index=False)
submission_elastic.to_csv('./dataset/submission_elastic_net.csv', index=False)
submission_ensemble.to_csv('./dataset/submission_ensemble.csv', index=False)

print("✓ Submission files created:")
print("  - submission_xgboost.csv")
print("  - submission_elastic_net.csv")
print("  - submission_ensemble.csv")

# Display sample
print(f"\n{'='*80}")
print("SAMPLE PREDICTIONS (First 10 houses)")
print("="*80)
sample_predictions = pd.DataFrame({
    'Id': test_df['Id'].head(10),
    'XGBoost': predictions_xgb[:10],
    'Elastic_Net': predictions_elastic[:10],
    'Ensemble': predictions_ensemble[:10]
})
print(sample_predictions.to_string(index=False))

<a id="section5"></a>
## 5. Prediction Analysis

In [ ]:
# Analyze prediction distribution
print("="*80)
print("PREDICTION DISTRIBUTION ANALYSIS")
print("="*80)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. XGBoost distribution
ax1 = axes[0, 0]
ax1.hist(predictions_xgb, bins=50, edgecolor='black', alpha=0.7, color='orange')
ax1.axvline(predictions_xgb.mean(), color='red', linestyle='--', 
           label=f'Mean: ${predictions_xgb.mean():,.0f}')
ax1.set_xlabel('Predicted Price ($)')
ax1.set_ylabel('Frequency')
ax1.set_title('XGBoost Predictions Distribution')
ax1.legend()
ax1.grid(alpha=0.3)

# 2. Elastic Net distribution
ax2 = axes[0, 1]
ax2.hist(predictions_elastic, bins=50, edgecolor='black', alpha=0.7, color='steelblue')
ax2.axvline(predictions_elastic.mean(), color='red', linestyle='--', 
           label=f'Mean: ${predictions_elastic.mean():,.0f}')
ax2.set_xlabel('Predicted Price ($)')
ax2.set_ylabel('Frequency')
ax2.set_title('Elastic Net Predictions Distribution')
ax2.legend()
ax2.grid(alpha=0.3)

# 3. Ensemble distribution
ax3 = axes[1, 0]
ax3.hist(predictions_ensemble, bins=50, edgecolor='black', alpha=0.7, color='green')
ax3.axvline(predictions_ensemble.mean(), color='red', linestyle='--', 
           label=f'Mean: ${predictions_ensemble.mean():,.0f}')
ax3.set_xlabel('Predicted Price ($)')
ax3.set_ylabel('Frequency')
ax3.set_title('Ensemble Predictions Distribution')
ax3.legend()
ax3.grid(alpha=0.3)

# 4. Model comparison
ax4 = axes[1, 1]
ax4.scatter(predictions_xgb, predictions_elastic, alpha=0.5, s=20)
ax4.plot([predictions_xgb.min(), predictions_xgb.max()], 
        [predictions_xgb.min(), predictions_xgb.max()], 'r--', lw=2)
ax4.set_xlabel('XGBoost Predictions ($)')
ax4.set_ylabel('Elastic Net Predictions ($)')
ax4.set_title('Model Agreement: XGBoost vs Elastic Net')
ax4.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\n✓ Prediction analysis visualizations complete")

In [ ]:
# Calculate model agreement metrics
print("\n" + "="*80)
print("MODEL AGREEMENT ANALYSIS")
print("="*80)

# Difference between models
difference = predictions_xgb - predictions_elastic
percent_diff = (difference / predictions_ensemble) * 100

print(f"\nPrediction Differences:")
print(f"  Mean absolute difference: ${np.abs(difference).mean():,.0f}")
print(f"  Max difference: ${np.abs(difference).max():,.0f}")
print(f"  Mean % difference: {np.abs(percent_diff).mean():.2f}%")

# Correlation between models
correlation = np.corrcoef(predictions_xgb, predictions_elastic)[0, 1]
print(f"\nCorrelation between models: {correlation:.4f}")

# Identify houses with largest disagreement
disagreement_df = pd.DataFrame({
    'Id': test_df['Id'],
    'XGBoost': predictions_xgb,
    'Elastic_Net': predictions_elastic,
    'Difference': np.abs(difference),
    'Percent_Diff': np.abs(percent_diff)
})

print(f"\n{'='*80}")
print("TOP 10 HOUSES WITH LARGEST MODEL DISAGREEMENT")
print("="*80)
print(disagreement_df.nlargest(10, 'Difference').to_string(index=False))

<a id="section6"></a>
## 6. Export Results

In [ ]:
# Create comprehensive results file
print("="*80)
print("CREATING COMPREHENSIVE RESULTS FILE")
print("="*80)

# Combine all predictions
results_df = pd.DataFrame({
    'Id': test_df['Id'],
    'XGBoost_Prediction': predictions_xgb,
    'ElasticNet_Prediction': predictions_elastic,
    'Ensemble_Prediction': predictions_ensemble,
    'Prediction_Difference': np.abs(predictions_xgb - predictions_elastic),
    'Percent_Difference': np.abs((predictions_xgb - predictions_elastic) / predictions_ensemble * 100)
})

# Add original features (optional - for analysis)
if 'OverallQual' in test_df.columns:
    results_df['OverallQual'] = test_df['OverallQual']
if 'GrLivArea' in test_df.columns:
    results_df['GrLivArea'] = test_df['GrLivArea']
if 'Neighborhood' in test_df.columns:
    results_df['Neighborhood'] = test_df['Neighborhood']

# Save comprehensive results
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
results_filename = f'./dataset/predictions_comprehensive_{timestamp}.csv'
results_df.to_csv(results_filename, index=False)

print(f"✓ Comprehensive results saved: {results_filename}")
print(f"\nResults include:")
print(f"  - All three model predictions")
print(f"  - Model agreement metrics")
print(f"  - Key house features")
print(f"\nTotal records: {len(results_df)}")

In [ ]:
# Summary statistics by price range
print("\n" + "="*80)
print("PREDICTIONS BY PRICE RANGE (Ensemble)")
print("="*80)

# Create price bins
results_df['Price_Range'] = pd.cut(results_df['Ensemble_Prediction'], 
                                   bins=[0, 100000, 150000, 200000, 250000, 500000],
                                   labels=['<$100K', '$100-150K', '$150-200K', '$200-250K', '>$250K'])

price_range_summary = results_df.groupby('Price_Range').agg({
    'Id': 'count',
    'Ensemble_Prediction': ['mean', 'min', 'max'],
    'Prediction_Difference': 'mean'
}).round(0)

price_range_summary.columns = ['Count', 'Avg_Price', 'Min_Price', 'Max_Price', 'Avg_Model_Diff']
print(price_range_summary)

# Visualize
plt.figure(figsize=(10, 6))
price_range_summary['Count'].plot(kind='bar', color='teal', alpha=0.7)
plt.xlabel('Price Range')
plt.ylabel('Number of Houses')
plt.title('Predicted House Count by Price Range')
plt.xticks(rotation=45)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## Summary and Next Steps

### What We Accomplished:

1. ✓ Loaded trained models from Notebook 1
2. ✓ Preprocessed test data using the same pipeline as training
3. ✓ Generated predictions using XGBoost, Elastic Net, and Ensemble
4. ✓ Created submission files for all three models
5. ✓ Analyzed prediction distributions and model agreement
6. ✓ Exported comprehensive results with metadata

### Files Created:

- `submission_xgboost.csv` - XGBoost predictions
- `submission_elastic_net.csv` - Elastic Net predictions
- `submission_ensemble.csv` - Ensemble predictions (recommended)
- `predictions_comprehensive_[timestamp].csv` - Full results with analysis

### Model Performance Notes:

- **XGBoost**: Typically better at capturing non-linear relationships
- **Elastic Net**: More interpretable, good for linear trends
- **Ensemble**: Combines strengths of both, usually most robust

### Recommendations:

1. **Use the Ensemble predictions** for submission (balanced performance)
2. **Review houses with large model disagreement** - may indicate uncertainty
3. **Check predictions against domain knowledge** - do they make sense?
4. **Consider outliers** - very high/low predictions may need review

### Next Steps:

- Submit `submission_ensemble.csv` for evaluation
- Use Notebook 2 to analyze individual predictions with SHAP
- Review houses with unusual predictions
- Consider retraining with different parameters if needed

---

**Notebook Complete! All predictions have been generated successfully.**